# E1.3 — The encoder ladder: C.11 as a falsifiable prediction

Section C.11 explains the GPT-2 arm's collapse-and-rescue with one
mechanism: *what similarity training supplies is isotropy, not
correspondence*. If that mechanism is right, it predicts the outcome for
encoders it has never seen. This notebook pre-registers those
predictions, then measures.

The ladder spans the training-objective spectrum, all evaluated as
targets for the SAME image encoder (DINOv2-base, cls+patch, the cached
E1 embeddings) under the SAME protocol:

| encoder | objective | predicted eff. rank | predicted raw R@1 | predicted after whitening |
|---|---|---|---|---|
| GPT-2 *(measured)* | causal LM | 6.5 ✓ | 0.160 ✓ | 0.478 ✓ |
| BERT, mean-pooled | masked LM, bidirectional | low-to-middling | between GPT-2 and SBERT | ≈ bge level |
| SBERT (all-mpnet-base-v2) | MLM + contrastive | high | near bge | little further gain |
| bge-m3 *(measured)* | heavy contrastive | high ✓ | 0.466 ✓ | — |

**Pre-registered claims, falsifiable by the cells below:**
1. Raw R@1 orders by the target space's effective rank (GPT-2 < BERT <
   SBERT ≈ bge), NOT by "amount of language understanding".
2. Whitening collapses the ladder: every encoder lands near the bge arm
   regardless of objective, and the contrastive ones gain little
   (they are already isotropic).
3. If BERT lands OUT of rank order, the isotropy account is incomplete
   — that is a reportable limit, not a failure of the notebook.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"])
import torch
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print("DATA_DIR:", DATA_DIR, "| device:", DEVICE)

In [ ]:
import numpy as np, json, zipfile

# image side: the SAME cached base embeddings the measured rows used
IMG_CKPT = DATA_DIR / "e1_img_ckpt_dinov2-base_cls+patch.npz"
assert IMG_CKPT.exists(), "run E1 (base, cls+patch) first"
IMG_ALL = np.load(str(IMG_CKPT))["img"].astype(np.float64)
N_PAIRS = len(IMG_ALL)
print(f"image cache: {IMG_ALL.shape}  "
      f"({(N_PAIRS-1000)/IMG_ALL.shape[1]:.1f} rows/dim after eval split)")

# captions: E1's exact recipe, ALL_CAPTIONS averaged after encoding
zf = DATA_DIR / "annotations_trainval2017.zip"
assert zf.exists(), "annotations zip missing"
with zipfile.ZipFile(str(zf)) as z:
    ann = json.load(z.open("annotations/captions_train2017.json"))
url  = {im["id"]: im["coco_url"] for im in ann["images"]}
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())
ids = sorted(set(caps) & set(url))[:N_PAIRS]
assert len(ids) == N_PAIRS
captions = [caps[i] for i in ids]          # list of ~5 strings per image
print(f"{len(ids)} caption groups, "
      f"{np.mean([len(c) for c in captions]):.1f} captions/image")

# measured reference rows for this image configuration (base, cls+patch)
MEASURED = {
    "gpt2":  dict(er=6.5,  raw_r1=0.160, whit_r1=0.478),
    "bge-m3": dict(r2=0.580, r1=0.466),
}
MAX_LEN = 64
rng = np.random.default_rng(0)
perm = rng.permutation(N_PAIRS)
te, tr = perm[:1000], perm[1000:]

In [ ]:
# ---- encode captions with each ladder encoder (cached per encoder) ----
from transformers import AutoTokenizer, AutoModel

LADDER = {
    # short   HF checkpoint                                   pooling
    "bert":  ("bert-base-uncased",                            "mean"),
    "sbert": ("sentence-transformers/all-mpnet-base-v2",      "mean"),
}
# (SBERT's own inference for this checkpoint IS attention-masked mean
#  pooling, so loading it through AutoModel + mean pool is faithful.)

def encode_texts(ckpt, pooling, tag):
    out = DATA_DIR / f"e13_txt_{tag}.npz"       # cache-key carries encoder
    if out.exists():
        arr = np.load(str(out))["txt"]
        if len(arr) == N_PAIRS:
            print(f"  {tag}: cached {arr.shape}")
            return arr.astype(np.float64)
        print(f"  {tag}: cache has {len(arr)} rows, need {N_PAIRS} - redoing")
    tok = AutoTokenizer.from_pretrained(ckpt)
    mod = AutoModel.from_pretrained(ckpt).to(DEVICE).eval()
    flat = [c for group in captions for c in group]
    sizes = [len(group) for group in captions]
    vecs = []
    B = 256
    with torch.no_grad():
        for i in range(0, len(flat), B):
            b = tok(flat[i:i+B], padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
            h = mod(**b).last_hidden_state           # [B, T, d]
            m = b["attention_mask"].unsqueeze(-1)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)   # masked mean
            vecs.append(v.float().cpu().numpy())
    V = np.concatenate(vecs)
    # average the ~5 caption embeddings per image, as everywhere else
    out_rows, k = [], 0
    for s in sizes:
        out_rows.append(V[k:k+s].mean(0)); k += s
    T = np.stack(out_rows)
    np.savez_compressed(str(out), txt=T.astype(np.float32))
    print(f"  {tag}: encoded {T.shape}, cached")
    del mod
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return T.astype(np.float64)

TXT = {tag: encode_texts(ck, pl, tag) for tag, (ck, pl) in LADDER.items()}

In [ ]:
# ---- the identical protocol, raw and whitened, per encoder ----
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
def r2s(Y, P):
    return float(1 - ((Y-P)**2).sum() / ((Y-Y.mean(0))**2).sum())
def recall1(P, G):
    return float(((l2n(P) @ l2n(G).T).argmax(1) == np.arange(len(P))).mean())
def effrank(V):
    Vc = V - V.mean(0)
    s = np.linalg.svd(Vc, full_matrices=False, compute_uv=False)
    p = s**2 / (s**2).sum(); p = p[p > 0]
    return float(np.exp(-(p * np.log(p)).sum()))
def paircos(V):
    Vn = l2n(V); r = np.random.default_rng(1)
    i, j = r.integers(0, len(V), 3000), r.integers(0, len(V), 3000)
    m = i != j
    return float((Vn[i[m]] * Vn[j[m]]).sum(1).mean())
def whiten_fit(Xtr):
    mu = Xtr.mean(0)
    C = np.cov((Xtr-mu).T) + 1e-6*np.eye(Xtr.shape[1])
    ev, V = np.linalg.eigh(C); ev = np.clip(ev, 1e-8, None)
    W = V @ np.diag(ev**-0.5) @ V.T
    return lambda X: (X - mu) @ W

def arm(TGT, label):
    er, pc = effrank(TGT), paircos(TGT)
    best = (-1, None)
    for a in (1e-3, 1e-2, 1e-1, 1.0):
        Wm = ridge(IMG_ALL[tr][:-1000], TGT[tr][:-1000], a)
        v = r2s(TGT[tr][-1000:], IMG_ALL[tr][-1000:] @ Wm)
        if v > best[0]: best = (v, a)
    a = best[1]
    W = ridge(IMG_ALL[tr], TGT[tr], a)
    P = IMG_ALL[te] @ W
    raw = dict(r2=r2s(TGT[te], P), r1=recall1(P, TGT[te]))
    wf = whiten_fit(TGT[tr])                      # train targets ONLY
    Ww = ridge(IMG_ALL[tr], wf(TGT[tr]), a)
    Pw = IMG_ALL[te] @ Ww
    whit = dict(r2=r2s(wf(TGT[te]), Pw), r1=recall1(Pw, wf(TGT[te])),
                er=effrank(wf(TGT[tr])))
    print(f"{label:6s} er {er:6.1f}  pair-cos {pc:+.3f}  |  "
          f"raw R2 {raw['r2']:.3f} R@1 {raw['r1']:.3f}  |  "
          f"whitened R@1 {whit['r1']:.3f} (er {whit['er']:.0f})")
    return dict(er=er, pc=pc, raw=raw, whit=whit)

print("measured anchors: gpt2 er 6.5, raw R@1 0.160, whitened 0.478;")
print("                  bge-m3 R@1 0.466 (raw; already isotropic)\n")
RES = {tag: arm(T, tag) for tag, T in TXT.items()}

In [ ]:
# ---- score the pre-registered predictions ----
ladder = [("gpt2", 6.5, 0.160, 0.478)] + \
         [(t, RES[t]["er"], RES[t]["raw"]["r1"], RES[t]["whit"]["r1"])
          for t in RES]
BGE = 0.466
print("encoder      eff.rank   raw R@1   whitened R@1   % of bge (whit)")
for name, er, r1, w1 in ladder:
    print(f"{name:10s} {er:9.1f} {r1:9.3f} {w1:13.3f} {100*w1/BGE:12.1f}%")
print(f"{'bge-m3':10s} {'high':>9s} {BGE:9.3f} {'-':>13s} {100.0:12.1f}%")

ers = [x[1] for x in ladder]; raws = [x[2] for x in ladder]
rank = lambda v: np.argsort(np.argsort(v))
rho = float(np.corrcoef(rank(ers), rank(raws))[0, 1])
whits = [x[3] for x in ladder]
spread = max(whits) - min(whits)
print(f"\nPREDICTION 1 - raw R@1 orders by effective rank: "
      f"rank-corr {rho:+.2f} "
      + ("SUPPORTED" if rho > 0.6 else "NOT SUPPORTED"))
print(f"PREDICTION 2 - whitening collapses the ladder: "
      f"whitened spread {spread:.3f} "
      + ("SUPPORTED (tight)" if spread < 0.08 else "PARTIAL/NOT - report it"))
print("\nEither outcome is a result: support turns C.11 into a confirmed")
print("prediction; violation locates the limit of the isotropy account.")